In [ ]:
#Setup and import

import pandas as pd
import numpy as np
import unicodedata
import hashlib

from pathlib import Path
from collections import Counter


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
#Paths

PROJECT_DIR = Path("/content/drive/MyDrive/Final_Project/")
DATA_DIR = PROJECT_DIR / "data"
SPLITS_DIR = DATA_DIR / "final_splits"

#data

SYN_TRAIN_PATH   = SPLITS_DIR / "syn_train.csv"
REAL_TRAIN_PATH  = SPLITS_DIR / "mixederr_train.csv"
GOLD_VAL_PATH    = SPLITS_DIR / "gold_eval.csv"
GOLD_TEST_PATH   = SPLITS_DIR / "gold_test.csv"

#labels

LABELS = [
    "verb_form",
    "tense_mood_aspect",
    "lexical_choice",
    "agreement",
    "word_order",
    "pronoun_clitic",
    "preposition",
    "determiner_article",
    "negation",
    "spelling_orthography"
]

In [ ]:
#load the datasets

train_syn = pd.read_csv(SYN_TRAIN_PATH)
train_real = pd.read_csv(REAL_TRAIN_PATH)
gold_val = pd.read_csv(GOLD_VAL_PATH)
gold_test = pd.read_csv(GOLD_TEST_PATH)

datasets = {
            "train_syn": train_syn,
            "train_real": train_real,
            "gold_val": gold_val,
            "gold_test": gold_test
            }

In [ ]:
#Basic sanity checks (structure)

for name, df in datasets.items():
    print(f"=== {name} ===")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Dtypes:")
    print(df.dtypes)
    print()


=== train_syn ===
Shape: (14494, 13)
Columns: ['id', 'clean', 'learner_sentence', 'label', 'agreement_subtype', 'tier', 'rule', 'rule_id', 'notes', 'spelling_subtype', 'idx', 'decision', 'reject_reason']
Dtypes:
id                    object
clean                 object
learner_sentence      object
label                 object
agreement_subtype     object
tier                  object
rule                  object
rule_id               object
notes                float64
spelling_subtype      object
idx                  float64
decision              object
reject_reason        float64
dtype: object

=== train_real ===
Shape: (398, 7)
Columns: ['idx', 'clean', 'learner_sentence', 'label', 'agreement_subtype', 'REPLACE', 'notes']
Dtypes:
idx                    int64
clean                 object
learner_sentence      object
label                 object
agreement_subtype     object
REPLACE              float64
notes                 object
dtype: object

=== gold_val ===
Shape: (183, 5)
Column

In [ ]:
#checking missing value

for name, df in datasets.items():
  print (f"==={name}===")
  print ("missing values")
  print (df.isna().sum())

#raw leakage inspection

for name, df in datasets.items():
  print (f"==={name}===")
  print ("raw leakage inspection")
  print (df.duplicated(subset=["learner_sentence", "label"]).sum())


===train_syn===
missing values
id                       0
clean                  154
learner_sentence         0
label                    0
agreement_subtype     7351
tier                  6707
rule                  4521
rule_id              13110
notes                14494
spelling_subtype     14254
idx                   1003
decision             13012
reject_reason        14494
dtype: int64
===train_real===
missing values
idx                    0
clean                  0
learner_sentence       0
label                  0
agreement_subtype    346
REPLACE              235
notes                235
dtype: int64
===gold_val===
missing values
learner_sentence       0
correction_1          70
label                  0
agreement_subtype    160
notes                 33
dtype: int64
===gold_test===
missing values
learner_sentence       0
correction_1          37
label                  0
agreement_subtype    239
notes                 26
REPLACE               76
__src__               76
dtype: int6

In [ ]:
#check the type of duplicates and drop on train_syn
dup = (
    datasets["train_syn"]
    .groupby("learner_sentence")["label"]
    .nunique()
    .sort_values(ascending=False)
)

dup[dup > 1].head(10)


#drop the duplicates

datasets["train_syn"] = datasets["train_syn"].drop_duplicates(
    subset=["learner_sentence", "label"]
)


#shape
datasets["train_syn"].shape


(14341, 13)

In [ ]:
#drop unuseful columns

KEEP_COLS = ["learner_sentence", "label"]

for name, df in datasets.items():
    datasets[name] = df[KEEP_COLS].copy()

for name, df in datasets.items():
  print (df.columns)

Index(['learner_sentence', 'label'], dtype='object')
Index(['learner_sentence', 'label'], dtype='object')
Index(['learner_sentence', 'label'], dtype='object')
Index(['learner_sentence', 'label'], dtype='object')


In [ ]:
#cheking labels distribution

for name, df in datasets.items():
  print (f"==={name}===")
  print (df["label"].value_counts())

===train_syn===
label
verb_form               1837
tense_mood_aspect       1674
lexical_choice          1613
agreement               1596
word_order              1482
pronoun_clitic          1439
preposition             1384
negation                1206
spelling_orthography    1056
determiner_article      1054
Name: count, dtype: int64
===train_real===
label
verb_form               74
agreement               52
preposition             48
spelling_orthography    46
determiner_article      43
negation                40
lexical_choice          32
tense_mood_aspect       23
word_order              20
pronoun_clitic          20
Name: count, dtype: int64
===gold_val===
label
verb_form               32
lexical_choice          31
agreement               23
preposition             20
spelling_orthography    19
determiner_article      13
tense_mood_aspect       13
negation                12
word_order              10
pronoun_clitic          10
Name: count, dtype: int64
===gold_test===
label
verb

In [ ]:

# if no weird labels
expected_labels = set (LABELS)
actual_labels = {label for df in datasets.values() for label in df["label"].dropna().unique()}

unexpected_labels = actual_labels - expected_labels
missing_labels = expected_labels - actual_labels

assert not unexpected_labels, f"Unexpected Labels Found: {unexpected_labels }"
assert not missing_labels, f"Missing Labels Found: {missing_labels }"

print ("NO UNEXPECTED OR MISSING LABELS FOUND")

#identify outliers in text : long vs short

for name, df in datasets.items():
  print (f"===={name}====")
  df["text_length"] = df["learner_sentence"].fillna("").str.split().str.len()
  print (f"Short junk:", df[df["text_length"] < 2].head())
  print (f"Very long:", df[df["text_length"] > 500].head())
  print (df["text_length"].describe())


NO UNEXPECTED OR MISSING LABELS FOUND
====train_syn====
Short junk: Empty DataFrame
Columns: [learner_sentence, label, text_length]
Index: []
Very long: Empty DataFrame
Columns: [learner_sentence, label, text_length]
Index: []
count    14341.000000
mean        19.026219
std          7.472265
min          3.000000
25%         13.000000
50%         18.000000
75%         24.000000
max         51.000000
Name: text_length, dtype: float64
====train_real====
Short junk: Empty DataFrame
Columns: [learner_sentence, label, text_length]
Index: []
Very long: Empty DataFrame
Columns: [learner_sentence, label, text_length]
Index: []
count    398.000000
mean      12.974874
std        4.755416
min        5.000000
25%       10.000000
50%       13.000000
75%       15.000000
max       51.000000
Name: text_length, dtype: float64
====gold_val====
Short junk: Empty DataFrame
Columns: [learner_sentence, label, text_length]
Index: []
Very long: Empty DataFrame
Columns: [learner_sentence, label, text_length]
I

In [ ]:
for name, df in datasets.items():
    datasets[name] = df.drop(columns=["text_length"])

In [ ]:
for name, df in datasets.items():
  print (df.columns)

Index(['learner_sentence', 'label'], dtype='object')
Index(['learner_sentence', 'label'], dtype='object')
Index(['learner_sentence', 'label'], dtype='object')
Index(['learner_sentence', 'label'], dtype='object')


In [ ]:
#Empty / whitespace-only sentences

for name, df in datasets.items():
  datasets[name]= df[df["learner_sentence"].fillna("").str.strip().ne("")]

#Unicode / encoding issues

for name, df in datasets.items():
    datasets[name]["learner_sentence"] = (
        df["learner_sentence"]
        .astype(str)
        .apply(lambda x: unicodedata.normalize("NFC", x))
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

#dedup
for name, df in datasets.items():
    datasets[name] = df.drop_duplicates(subset=["learner_sentence", "label"])

In [ ]:
for k in datasets.keys():
    print(repr(k))


'train_syn'
'train_real'
'gold_val'
'gold_test'


In [ ]:

def hash_text(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()

for name, df in datasets.items():
    datasets[name]["text_id"] = df["learner_sentence"].apply(hash_text)

train_syn_set  = set(datasets["train_syn"]["text_id"])
train_real_set = set(datasets["train_real"]["text_id"])
gold_val_set   = set(datasets["gold_val"]["text_id"])
gold_test_set  = set(datasets["gold_test"]["text_id"])

assert train_syn_set.isdisjoint(train_real_set)
assert train_syn_set.isdisjoint(gold_val_set)
assert train_syn_set.isdisjoint(gold_test_set)
assert train_real_set.isdisjoint(gold_val_set)
assert train_real_set.isdisjoint(gold_test_set)
assert gold_val_set.isdisjoint(gold_test_set)

print("NO DATA LEAKAGE ACROSS SPLITS")


NO DATA LEAKAGE ACROSS SPLITS


In [ ]:
for name, df in datasets.items():
    datasets[name] = df.drop(columns=["text_id"])

In [ ]:
#freeze the datasets

CLEAN_DIR = DATA_DIR / "final_splits_clean"
CLEAN_DIR.mkdir(exist_ok=True)

datasets["train_syn"].to_csv(CLEAN_DIR / "train_syn_clean.csv", index=False)
datasets["train_real"].to_csv(CLEAN_DIR / "train_real_clean.csv", index=False)
datasets["gold_val"].to_csv(CLEAN_DIR / "gold_val_clean.csv", index=False)
datasets["gold_test"].to_csv(CLEAN_DIR / "gold_test_clean.csv", index=False)


check = pd.read_csv(CLEAN_DIR / "train_syn_clean.csv")
print(f"===syn dataset===")
print(check.shape)
print(check.columns)
print(check["label"].value_counts().head(10))


check = pd.read_csv(CLEAN_DIR / "train_real_clean.csv")
print(f"===real dataset===")
print(check.shape)
print(check.columns)
print(check["label"].value_counts().head(10))


check = pd.read_csv(CLEAN_DIR / "gold_val_clean.csv")
print(f"===val dataset===")
print(check.shape)
print(check.columns)
print(check["label"].value_counts().head(10))


check = pd.read_csv(CLEAN_DIR / "gold_test_clean.csv")
print(f"===test dataset===")
print(check.shape)
print(check.columns)
print(check["label"].value_counts().head(10))




===syn dataset===
(14341, 2)
Index(['learner_sentence', 'label'], dtype='object')
label
verb_form               1837
tense_mood_aspect       1674
lexical_choice          1613
agreement               1596
word_order              1482
pronoun_clitic          1439
preposition             1384
negation                1206
spelling_orthography    1056
determiner_article      1054
Name: count, dtype: int64
===real dataset===
(398, 2)
Index(['learner_sentence', 'label'], dtype='object')
label
verb_form               74
agreement               52
preposition             48
spelling_orthography    46
determiner_article      43
negation                40
lexical_choice          32
tense_mood_aspect       23
word_order              20
pronoun_clitic          20
Name: count, dtype: int64
===val dataset===
(183, 2)
Index(['learner_sentence', 'label'], dtype='object')
label
verb_form               32
lexical_choice          31
agreement               23
preposition             20
spelling_orthograph